In [13]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from torchvision import transforms
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
from dataclasses import dataclass
from torch.nn import functional as F
import copy
from torchvision.models import resnet50, ResNet50_Weights
import random

torch.manual_seed(0)
random.seed(0)
np.random.seed(0)

# Split and load dataset

In [ ]:
dataset = load_dataset("microsoft/cats_vs_dogs")["train"]

splits = dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = splits["train"]
test_dataset = splits["test"]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    #transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

def train_transforms(examples):
    return {
        "pixel_values": [
            train_transform(image.convert("RGB"))
            for image in examples["image"]
        ],
        "labels": examples["labels"],
    }

def test_transforms(examples):
    return {
        "pixel_values": [
            test_transform(image.convert("RGB"))
            for image in examples["image"]
        ],
        "labels": examples["labels"],
    }

train_dataset = train_dataset.with_transform(train_transforms)
test_dataset = test_dataset.with_transform(test_transforms)

In [ ]:
def collate_as_tuple(samples):
    return (
        torch.stack([sample["pixel_values"] for sample in samples]),
        torch.tensor([sample["labels"] for sample in samples], dtype=torch.long),
    )


def cache_images(dataset, batch_size=128):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=0, collate_fn=collate_as_tuple)
    # Preallocate instead of keeping a list and torch.cat-ing: avoids a second
    # complete copy of all images at the end of cache creation.
    images = torch.empty((len(dataset), 3, 224, 224), dtype=torch.float32)
    labels = torch.empty(len(dataset), dtype=torch.long)
    offset = 0
    for x, y in loader:
        end = offset + len(y)
        images[offset:end].copy_(x)
        labels[offset:end].copy_(y)
        offset = end
    assert offset == len(dataset)
    return TensorDataset(images, labels)


def make_loader(dataset, batch_size, shuffle=False, device=None):
    cuda = (torch.device(device).type == "cuda" if device is not None
            else torch.cuda.is_available())
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=cuda, drop_last=False)


train_dataset = cache_images(train_dataset)
test_dataset = cache_images(test_dataset)
train_loader = make_loader(train_dataset, batch_size=512, shuffle=True)
test_loader = make_loader(test_dataset, batch_size=512)
print("Cached training images:", train_dataset.tensors[0].shape)
print("Cached test images:", test_dataset.tensors[0].shape)

Cached training images: torch.Size([18728, 3, 224, 224])
Cached test images: torch.Size([4682, 3, 224, 224])


# Simple MLP based tree network

In [16]:
@dataclass
class Config:
    epochs: int = 100
    refit_epochs: int = 20
    batch_size: int = 1024
    lr: float = 1e-2
    max_depth: int = 4
    device: str = "cpu"

    def __post_init__(self):
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
        elif torch.backends.mps.is_available():
            self.device = torch.device("mps")
        else:
            self.device = torch.device("cpu")

cfg = Config()

In [17]:
class TreeNode(nn.Module):
    def __init__(self,image_dim):
        super().__init__()
        self.trunk = nn.Sequential(
            # 3 x 224 x 224
            nn.Conv2d(3, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.MaxPool2d(2),

            nn.Conv2d(8, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.MaxPool2d(2),
            # 16 x 14 x 14

            nn.Conv2d(8, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # 8*7*7

            nn.Flatten(),
            nn.Linear(8 * 7 * 7, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        self.left_branch, self.right_branch = None, None
        self.left_wgt, self.right_wgt = nn.Parameter(torch.tensor(0.0)), nn.Parameter(torch.tensor(0.0))

    def forward(self, x):
        out = self.trunk(x).squeeze(-1)
        if self.left_branch is not None:
            out += self.left_wgt.clamp_min(0.0)*(self.left_branch(x) >= 0).to(x.dtype)
        if self.right_branch is not None:
            out -= self.right_wgt.clamp_min(0.0)*(self.right_branch(x) >= 0).to(x.dtype)
        return out

In [18]:
def ordered_loader(loader, cfg):
    return DataLoader(
        loader.dataset, batch_size=cfg.batch_size, shuffle=False,
        drop_last=False, collate_fn=loader.collate_fn,
    )

def dataset_labels(dataset):
    if isinstance(dataset, TensorDataset):
        return dataset.tensors[1]
    if isinstance(dataset, ChildDataset):
        return dataset.labels
    raise TypeError("Expected TensorDataset or ChildDataset")

In [19]:
@torch.no_grad()
def evaluate(node, loader, cfg, pos_weight=None):
    node.eval()
    count = 0
    errors = torch.zeros((), dtype=torch.long, device=cfg.device)
    loss_sum = torch.zeros((), device=cfg.device)
    for x, y in loader:
        x = x.to(cfg.device)
        y = y.to(cfg.device, dtype=torch.float32).reshape(-1)
        logits = node(x)
        count += y.numel()
        errors += ((logits >= 0) != y.bool()).sum()
        loss_sum += F.binary_cross_entropy_with_logits(
            logits, y,
            pos_weight=pos_weight,
            reduction="sum",
        )
    errors = errors.item()
    loss_sum = loss_sum.item()
    return {"errors": errors, "accuracy": 1 - errors / count,
            "loss": loss_sum / count, "count": count}

In [ ]:
def fit_node(node, dataloader, cfg, name=""):
    node.to(cfg.device)

    for branch in (node.left_branch, node.right_branch):
        if branch is not None:
            branch.requires_grad_(False)
            branch.eval()

    params = list(node.trunk.parameters()) + [node.left_wgt, node.right_wgt]
    optimizer = torch.optim.Adam(params, lr=cfg.lr)
    scan = ordered_loader(dataloader, cfg)

    total, pos, neg = 0,0,0

    labels = dataset_labels(dataloader.dataset)
    total = labels.numel()
    pos = int((labels == 1).sum().item())

    neg = total-pos
    pos, neg = max(pos,1), max(neg,1)
    pos_weight = torch.tensor(neg/pos, dtype=torch.float32, device=cfg.device)

    with torch.no_grad():
        node.left_wgt.clamp_(min=0)
        node.right_wgt.clamp_(min=0)

    initial = evaluate(node, scan, cfg, pos_weight=pos_weight)
    best_loss = initial["loss"]
    patience_reference = best_loss
    best_state = copy.deepcopy(node.state_dict())
    
    early_stop = 0
    min_delta = 1e-4
    for epoch in range(getattr(cfg, "epochs", 10)):
        node.trunk.train()
        for x,y in dataloader:
            x = x.to(cfg.device)
            y = y.to(device=cfg.device, dtype=torch.float32).reshape(-1)
            optimizer.zero_grad(set_to_none=True)
            logits = node(x)
            loss = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                node.left_wgt.clamp_(min=0)
                node.right_wgt.clamp_(min=0)

        metrics = evaluate(node, scan, cfg, pos_weight=pos_weight)
        current_loss = metrics["loss"]
        
        if current_loss < best_loss:
            best_loss = current_loss
            best_state = copy.deepcopy(node.state_dict())
        
        if current_loss < patience_reference - min_delta:
            patience_reference = current_loss
            early_stop = 0
        else:
            early_stop += 1
        
        if early_stop >= 5 and epoch + 1 >= 10:
            print("Early Stopping")
            break

        print(f"{name} epoch {epoch+1}: training accuracy={metrics['accuracy']:.2%}, "
              f"errors={metrics['errors']}, weighted BCE={metrics['loss']:.4f}")
        
    node.load_state_dict(best_state)
    return node.eval()

In [21]:
class ChildDataset(Dataset):
    def __init__(self, parent_dataset, indices, labels):
        self.parent_dataset = parent_dataset
        self.indices = indices.tolist()
        self.labels = labels.long().cpu()

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        sample = self.parent_dataset[self.indices[index]]
        if isinstance(sample, dict):
            x = sample["pixel_values"]
        else:
            x, _ = sample
        return x, self.labels[index]

In [ ]:
@torch.no_grad()
def make_child_loaders(node, parent_loader, cfg):
    scan_loader = DataLoader(
        parent_loader.dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        drop_last=False,
        collate_fn=parent_loader.collate_fn,
    )

    node.eval()

    predictions, labels = [], []
    for x, target in scan_loader:
        score = node(x.to(cfg.device)).reshape(-1)
        predictions.append((score >= 0).cpu())
        labels.append(target.reshape(-1).cpu().bool())

    pred = torch.cat(predictions)
    y = torch.cat(labels)

    false_negative = ~pred & y
    false_positive = pred & ~y

    left_mask = ~(pred & y)
    right_mask = pred | y

    def build_loader(mask, targets):
        indices = mask.nonzero(as_tuple=True)[0]

        child_dataset = ChildDataset(
            parent_dataset=parent_loader.dataset,
            indices=indices,
            labels=targets[indices],
        )

        return DataLoader(
            child_dataset,
            batch_size=cfg.batch_size,
            shuffle=True,
            drop_last=False,
        )

    left_loader = (
        build_loader(left_mask, false_negative)
        if false_negative.any()
        else None
    )

    right_loader = (
        build_loader(right_mask, false_positive)
        if false_positive.any()
        else None
    )

    return left_loader, right_loader

In [ ]:
def fit_tree(dataloader, cfg, depth=0, name=""):
    node = TreeNode((128,128,3)).to(cfg.device)

    node = fit_node(node, dataloader, cfg, name=name)
    if(depth >= cfg.max_depth):
        return node.eval()

    left_loader, right_loader = make_child_loaders(node, dataloader, cfg)

    if left_loader is not None:
        node.left_branch = fit_tree(left_loader, cfg, depth+1, name=name+"-left")

    if right_loader is not None:
        node.right_branch = fit_tree(right_loader, cfg, depth+1, name=name+"-right")

    if left_loader is None and right_loader is None:
        return node.eval()

    node = fit_node(node, dataloader, cfg, name=name+" refit")
    return node.eval()

In [ ]:
tree = fit_tree(train_loader, cfg)
train_metrics = evaluate(tree, ordered_loader(train_loader, cfg), cfg)
test_metrics = evaluate(tree, test_loader, cfg)
node_count = sum(isinstance(module, TreeNode) for module in tree.modules())
print("Training:", train_metrics)
print("Test:", test_metrics)
print("Nodes:", node_count)

 epoch 1: training accuracy=61.68%, errors=7176, weighted BCE=0.6511
 epoch 2: training accuracy=67.97%, errors=5999, weighted BCE=0.6037
 epoch 3: training accuracy=72.51%, errors=5148, weighted BCE=0.5587
 epoch 4: training accuracy=74.11%, errors=4848, weighted BCE=0.5347
 epoch 5: training accuracy=76.56%, errors=4390, weighted BCE=0.5045
 epoch 6: training accuracy=76.11%, errors=4475, weighted BCE=0.5086
 epoch 7: training accuracy=76.46%, errors=4409, weighted BCE=0.4880
 epoch 8: training accuracy=78.89%, errors=3954, weighted BCE=0.4640
 epoch 9: training accuracy=77.95%, errors=4130, weighted BCE=0.4714
 epoch 10: training accuracy=79.79%, errors=3785, weighted BCE=0.4483
 epoch 11: training accuracy=79.67%, errors=3808, weighted BCE=0.4481
 epoch 12: training accuracy=80.01%, errors=3744, weighted BCE=0.4428
 epoch 13: training accuracy=81.18%, errors=3525, weighted BCE=0.4246
 epoch 14: training accuracy=81.55%, errors=3455, weighted BCE=0.4229
 epoch 15: training accuracy=

In [ ]:
total_params = sum(p.numel() for p in tree.parameters())
trainable_params = sum(
    p.numel() for p in tree.parameters() if p.requires_grad
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {total_params - trainable_params:,}")
